In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, rankdata
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import os

In [2]:
print('Chargement des données...')
x_train = pd.read_csv("../data/x_train_final_asAbTs5.csv", index_col=0)
y_train = pd.read_csv('../data/y_train_final_YYyFil7.csv', index_col=0)
x_test  = pd.read_csv('../data/x_test_final_fIrnA7Q.csv',  index_col=0)

print(f'x_train : {x_train.shape}  |  y_train : {y_train.shape}')
print(f'x_test  : {x_test.shape}')

Chargement des données...
x_train : (6076546, 10)  |  y_train : (6076546, 1)
x_test  : (2028750, 10)


In [3]:
x_train

,total_count,longitude_scaled,latitude_scaled,Precipitations,HauteurNeige,Temperature,ForceVent,day_of_week,month_of_year,hour
0,1,0.998417,0.996118,0.0,0.0,14.6,2.5,3,4,8
1,35,0.999222,0.996000,0.0,0.0,22.4,3.1,5,3,13
2,3,0.998371,0.996309,1.8,0.0,11.7,3.3,4,10,8
3,1,0.998804,0.996343,0.0,0.0,28.6,2.6,4,2,16
4,2,0.999126,0.996417,0.0,0.0,9.6,3.2,3,9,18
...,...,...,...,...,...,...,...,...,...,...
6076541,9,0.998397,0.996100,0.0,0.0,-0.3,2.3,5,7,18
6076542,55,0.998655,0.996093,0.0,0.0,17.0,6.0,6,12,10
6076543,9,0.999200,0.996001,0.0,0.0,11.5,5.1,3,5,12
6076544,14,0.998374,0.996076,0.0,0.0,13.6,3.5,4,11,7


In [5]:
x_train["hour"].value_counts()

hour
9     598697
14    596622
8     594690
10    584009
11    565575
13    560183
15    550682
16    510814
12    491668
17    488246
7     329269
18    203597
19      1969
6        525
Name: count, dtype: int64

In [ ]:
x_train["is_morning_rush"] = x_train["hour"].apply(lambda x: 1 if x>=8 and x<=9 else 0)

In [16]:
x_train.loc[(x_train["hour"]>=8) & (x_train["hour"] <= 9), ["hour" , "is_morning_rush"]]

,hour,is_morning_rush
0,8,1
2,8,1
12,8,1
13,8,1
19,8,1
...,...,...
6076524,8,1
6076527,8,1
6076529,8,1
6076530,9,1


In [19]:
x_train[(x_train["day_of_week"]==5) | (x_train["day_of_week"]==6)] # Week-End

,total_count,longitude_scaled,latitude_scaled,Precipitations,HauteurNeige,Temperature,ForceVent,day_of_week,month_of_year,hour,is_morning_rush
1,35,0.999222,0.996000,0.0,0.0,22.4,3.1,5,3,13,0
8,20,0.998997,0.995835,0.0,0.0,23.3,6.6,5,4,17,0
12,9,0.998347,0.996116,0.0,0.0,12.0,3.4,6,5,8,1
26,3,0.998416,0.995914,0.0,0.0,15.5,3.1,5,9,18,0
30,3,0.998991,0.996736,0.0,NaN,21.5,2.9,6,12,13,0
...,...,...,...,...,...,...,...,...,...,...,...
6076531,10,0.998347,0.996161,0.0,0.0,13.2,4.7,5,8,13,0
6076534,6,0.999279,0.996033,0.0,0.0,13.7,3.6,6,4,15,0
6076537,12,0.999048,0.996213,0.0,0.0,10.5,1.0,5,6,12,0
6076541,9,0.998397,0.996100,0.0,0.0,-0.3,2.3,5,7,18,0


In [20]:
x_train["is_week_end"] = x_train["day_of_week"].apply(lambda x : 1 if x==5 or x==6 else 0)

In [21]:
x_train

,total_count,longitude_scaled,latitude_scaled,Precipitations,HauteurNeige,Temperature,ForceVent,day_of_week,month_of_year,hour,is_morning_rush,is_week_end
0,1,0.998417,0.996118,0.0,0.0,14.6,2.5,3,4,8,1,0
1,35,0.999222,0.996000,0.0,0.0,22.4,3.1,5,3,13,0,1
2,3,0.998371,0.996309,1.8,0.0,11.7,3.3,4,10,8,1,0
3,1,0.998804,0.996343,0.0,0.0,28.6,2.6,4,2,16,0,0
4,2,0.999126,0.996417,0.0,0.0,9.6,3.2,3,9,18,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
6076541,9,0.998397,0.996100,0.0,0.0,-0.3,2.3,5,7,18,0,1
6076542,55,0.998655,0.996093,0.0,0.0,17.0,6.0,6,12,10,0,1
6076543,9,0.999200,0.996001,0.0,0.0,11.5,5.1,3,5,12,0,0
6076544,14,0.998374,0.996076,0.0,0.0,13.6,3.5,4,11,7,0,0


In [25]:
x_train[x_train["Precipitations"] > 0]

,total_count,longitude_scaled,latitude_scaled,Precipitations,HauteurNeige,Temperature,ForceVent,day_of_week,month_of_year,hour,is_morning_rush,is_week_end
2,3,0.998371,0.996309,1.8,0.0,11.7,3.3,4,10,8,1,0
7,9,0.999009,0.996438,0.4,0.0,15.2,2.6,2,11,15,0,0
23,1,0.998971,0.996592,0.6,0.0,10.8,8.8,2,7,10,0,0
33,20,0.998569,0.995779,0.2,0.0,17.4,4.1,4,10,17,0,0
35,11,0.998416,0.996078,23.6,0.0,20.5,1.9,5,1,17,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
6076508,1,0.998449,0.996037,1.0,0.0,20.4,4.1,6,2,15,0,1
6076510,3,0.999168,0.996087,0.6,0.0,10.4,6.7,3,7,14,0,0
6076511,14,0.998907,0.995870,0.2,0.0,6.4,2.9,6,9,18,0,1
6076516,2,0.998923,0.995906,0.2,0.0,17.6,2.9,4,3,15,0,0


In [26]:
x_train["HauteurNeige"].value_counts()

HauteurNeige
0.0    5887677
1.0      24853
Name: count, dtype: int64

In [30]:
x_train["Temperature"].max()

np.float64(35.3)

In [ ]:
df_temp = x_train["Temperature"].values

# Par mois
month_labels = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
monthly = df_temp.groupby('month_of_year')['invalid_ratio'].mean().reindex(range(1, 13), fill_value=0)
axes[2].bar(month_labels, monthly.values, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Invalid ratio moyen par mois', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Mois')
axes[2].set_ylabel('invalid_ratio moyen')